# 🍜 Analisis Klaster Pola Pengeluaran Makanan & Minuman Jadi di Jawa Timur 2024
**Mata Kuliah:** Pemodelan Statistik Terapan  
**Framing Industri:** FMCG & Consumer Analytics Insights (Regional Distribution & Pricing Strategy)  
**Metodologi:** Agglomerative Hierarchical Clustering (StandardScaler, Linkage Comparison, Cophenetic Correlation, Per-Sample Silhouette Analysis)

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
root_path = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import src
from src.clustering import (
    calculate_cophenetic_correlation,
    compare_linkage_methods,
    fit_hierarchical_clustering,
    compute_sample_silhouette,
    get_cluster_profiles,
    get_gap_analysis,
    assign_business_cluster_names,
    get_business_recommendations
)
from src.visualization import (
    plot_dendrogram,
    plot_silhouette_sample_analysis,
    plot_pca_2d_interactive,
    plot_cluster_distribution_interactive,
    plot_cluster_profile_heatmap_interactive,
    plot_top_features_interactive,
    plot_gap_analysis_interactive,
    plot_east_java_map_interactive
)

print('Libraries and custom modules successfully loaded.')

## 1. Data Ingestion & Inspeksi Awal
Dataset bersumber dari Badan Pusat Statistik (BPS) Jawa Timur yang mencatat rata-rata pengeluaran per kapita seminggu untuk komoditas makanan dan minuman jadi di seluruh 38 Kabupaten/Kota tahun 2024.

In [ ]:
df_raw = src.load_raw_data()
print(f'Dimensi Data Mentah: {df_raw.shape[0]} Wilayah x {df_raw.shape[1]} Kolom')
df_raw.head()

In [ ]:
df_clean, regions, df_numeric = src.clean_data(df_raw)
print(f'Fitur Numerik Teridentifikasi: {df_numeric.shape[1]} kolom')

## 2. Preprocessing & Justifikasi Metodologis
### 2.1 Justifikasi Statistik Penghapusan Fitur `Minuman keras`
Sebelum melakukan penghapusan fitur, kami melakukan pengujian empiris kuantitatif pada missing rate, rasio varians, dan korelasi.

In [ ]:
drop_diag = src.analyze_dropped_feature(df_numeric, 'Minuman keras')
print(drop_diag['justification'])

In [ ]:
# Drop fitur Minuman keras secara terjustifikasi
df_features = df_numeric.drop(columns=['Minuman keras'], errors='ignore')
feature_names = df_features.columns.tolist()
print(f'Jumlah fitur final untuk pemodelan: {len(feature_names)} komoditas')

### 2.2 Deteksi & Rationale Penanganan Outlier (IQR Method)
Karena data ini merupakan data sensus populasi (38 Kabupaten/Kota se-Jawa Timur), pencilan tinggi pada Kota Surabaya dan Kota Malang mencerminkan fenomena ekonomi nyata. Outlier dipertahankan untuk mendeteksi segmen metropolitan.

In [ ]:
outlier_res = src.detect_outliers_iqr(df_features, region_names=regions)
print(outlier_res['methodology_rationale'])
outlier_res['summary_table'].head(10)

### 2.3 Standardisasi Fitur (Z-score Normalization)
Menggunakan `StandardScaler` agar perbedaan skala nominal antar komoditas tidak mendominasi perhitungan jarak Euclidean.

In [ ]:
df_scaled, scaler = src.scale_features(df_features, index_series=regions)
df_scaled.head()

## 3. Eksplorasi Metodologis: Evaluasi 4 Metode Linkage
Kami mengevaluasi 4 metode linkage (`Ward`, `Complete`, `Average`, `Single`) berdasarkan Silhouette Score (K=2 s/d 10) dan *Cophenetic Correlation Coefficient*.

In [ ]:
scores_df, summary_eval = src.compare_linkage_methods(df_scaled.values)
summary_eval

In [ ]:
fig_linkage = src.visualization.plot_linkage_comparison_interactive(scores_df, chosen_method='ward', chosen_k=3)
fig_linkage.show()

## 4. Pemodelan Hierarchical Clustering (Ward, K=3)
Metode Ward dipilih karena menghasilkan klaster yang homogen dengan ukuran klaster yang berimbang secara manajerial.

In [ ]:
fig_dendro = plot_dendrogram(df_scaled.values, labels=regions.tolist(), method='ward')
plt.show()

In [ ]:
model, labels, sil_score = fit_hierarchical_clustering(df_scaled.values, n_clusters=3, linkage_method='ward')
print(f'Silhouette Score Ward (K=3): {sil_score:.4f}')

fig_sil_sample = plot_silhouette_sample_analysis(df_scaled.values, labels, n_clusters=3)
plt.show()

## 5. Visualisasi Hasil Klastering
### 5.1 Proyeksi 2D PCA & Distribusi Anggota

In [ ]:
median_prof_raw, _ = get_cluster_profiles(df_features, labels)
cluster_name_map = assign_business_cluster_names(median_prof_raw)

named_labels = [cluster_name_map.get(f'Klaster {lbl}', f'Klaster {lbl}') for lbl in labels]
df_result = df_features.copy()
df_result['Kabupaten/Kota'] = regions.values
df_result['Cluster_ID'] = labels
df_result['Nama_Klaster'] = named_labels
df_result['Total_Pengeluaran'] = df_features.sum(axis=1)

fig_pca = plot_pca_2d_interactive(df_scaled, labels, regions, cluster_name_map, df_raw_features=df_features)
fig_pca.show()

In [ ]:
fig_dist = plot_cluster_distribution_interactive(df_result, cluster_col='Nama_Klaster')
fig_dist.show()

### 5.2 Peta Persebaran Spasial Jawa Timur

In [ ]:
fig_map = plot_east_java_map_interactive(df_result, total_spending_col='Total_Pengeluaran')
fig_map.show()

## 6. Business Insights & FMCG Strategic Recommendations
### 6.1 Analisis Disparitas Pengeluaran (Gap Analysis)

In [ ]:
median_prof_named, mean_prof_named = get_cluster_profiles(df_features, labels, label_names=cluster_name_map)
gap_df = get_gap_analysis(median_prof_named)
gap_df.head(10)

In [ ]:
fig_gap = plot_gap_analysis_interactive(gap_df, top_n=10)
fig_gap.show()

### 6.2 Top Komoditas & Rekomendasi Aksi Industri

In [ ]:
fig_top = plot_top_features_interactive(median_prof_named, top_n=10)
fig_top.show()

In [ ]:
recs = get_business_recommendations()
for cluster_title, rec_body in recs.items():
    print('=' * 80)
    print(f'SEGMEN: {cluster_title}')
    print(f'WHO: {rec_body["who"]}')
    print(f'WHAT: {rec_body["what"]}')
    print(f'SO WHAT: {rec_body["so_what"]}')
    print('NOW WHAT:')
    print(f'  - FMCG Strategy: {rec_body["now_what"]["fmcg"]}')
    print(f'  - Food Delivery: {rec_body["now_what"]["food_delivery"]}')
    print(f'  - Kebijakan Pemda: {rec_body["now_what"]["pemprov"]}')

## 7. Ekspor Data Hasil Segmentasi

In [ ]:
output_path = Path('data/processed/hasil_klaster_konsumsi_jatim_2024.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_result.to_csv(output_path, index=False)
print(f'Data hasil segmentasi berhasil diekspor ke: {output_path}')